# NYC Airbnb — EDA & Preprocessing

**Team 005 — CSE 6242 Spring 2026**

Builds the XGBoost-ready feature matrix from all raw datasets:
1. Load enriched Nov 2025 base (spatial features from notebook 01) + merge extra columns from full listings
2. Add LIRR proximity features
3. Engineer features: bathrooms, amenities, host stats, review scores, categoricals
4. EDA with Plotly: price heatmap, distributions, feature correlations
5. Save final XGBoost feature matrix

**Output:** `data/processed/xgboost_features.csv`

In [ ]:
import gzip
import re
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

warnings.filterwarnings('ignore')

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')
REPORTS   = Path('../reports')
REPORTS.mkdir(exist_ok=True)

print('Ready.')

---
## 1. Load Base Dataset & Merge Extra Columns

In [ ]:
# Load enriched dataset (spatial features already computed in notebook 01)
enriched = pd.read_csv(PROCESSED / 'listings_nov2025_enriched.csv')
print(f'Enriched base: {enriched.shape}')

# Load full Nov 2025 listings to get extra columns not in enriched
EXTRA_COLS = [
    'id', 'property_type', 'bathrooms_text', 'amenities',
    'host_is_superhost', 'host_response_time', 'host_response_rate',
    'host_acceptance_rate', 'host_identity_verified',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    'instant_bookable', 'license',
    'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    'first_review', 'last_review',
]

with gzip.open(RAW / 'listings_2025-11-01.csv.gz', 'rt') as f:
    full = pd.read_csv(f, low_memory=False, usecols=EXTRA_COLS)

df = enriched.merge(full, on='id', how='left')
print(f'After merge: {df.shape}')
df.head(2)

---
## 2. LIRR Proximity Features

In [ ]:
def haversine_matrix(lat1, lon1, lat2, lon2, chunk=500):
    """Vectorised haversine: returns (n_listings, n_ref) distance matrix in km, chunked."""
    R = 6371.0
    a1 = np.radians(lat1); o1 = np.radians(lon1)
    a2 = np.radians(lat2); o2 = np.radians(lon2)
    min_d = np.full(len(a1), np.inf)
    cnt   = np.zeros(len(a1), dtype=int)
    MILE_KM = 1.60934

    for i in range(0, len(a1), chunk):
        la = a1[i:i+chunk, None]; lo = o1[i:i+chunk, None]
        dlat = a2[None,:] - la; dlon = o2[None,:] - lo
        a = np.sin(dlat/2)**2 + np.cos(la)*np.cos(a2[None,:])*np.sin(dlon/2)**2
        d = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        min_d[i:i+chunk] = d.min(axis=1)
        cnt[i:i+chunk]   = (d <= MILE_KM).sum(axis=1)
    return min_d, cnt

lirr = pd.read_csv(RAW / 'lirr_stations.csv')
lirr_only = lirr[lirr['Railroad'] == 'LIRR'].dropna(subset=['Latitude','Longitude'])
print(f'LIRR stations: {len(lirr_only)}')

dist_lirr, cnt_lirr = haversine_matrix(
    df['latitude'].values, df['longitude'].values,
    lirr_only['Latitude'].values, lirr_only['Longitude'].values
)
df['dist_lirr_km']    = dist_lirr
df['lirr_count_1mi']  = cnt_lirr

print('dist_lirr_km stats:')
print(df['dist_lirr_km'].describe().round(2))
print(f"\nListings within 1 mi of LIRR: {(df['dist_lirr_km'] <= 1.609).sum():,}")

---
## 3. Feature Engineering

In [ ]:
# --- Bathrooms ---------------------------------------------------------------
def parse_bathrooms(s):
    """'1.5 baths' -> 1.5, 'Half-bath' -> 0.5, NaN -> NaN"""
    if pd.isna(s): return np.nan
    s = str(s).lower()
    if 'half' in s or 'private half' in s: return 0.5
    m = re.search(r'[\d\.]+', s)
    return float(m.group()) if m else np.nan

df['bathrooms'] = df['bathrooms_text'].apply(parse_bathrooms)
bath_median = df['bathrooms'].median()
df['bathrooms'] = df['bathrooms'].fillna(bath_median)
print('Bathrooms:', df['bathrooms'].value_counts().head())

In [ ]:
# --- Amenities ---------------------------------------------------------------
KEY_AMENITIES = {
    'has_wifi':    r'wifi|internet',
    'has_kitchen': r'kitchen',
    'has_ac':      r'air conditioning',
    'has_gym':     r'gym|fitness',
    'has_parking': r'parking',
    'has_pool':    r'pool',
    'has_dishwasher': r'dishwasher',
    'has_washer':  r'washer',
    'has_dryer':   r'dryer',
    'has_elevator': r'elevator',
    'has_doorman': r'doorman',
    'has_pets_allowed': r'pets allowed',
}

def parse_amenities(s):
    if pd.isna(s): return 0, {k: 0 for k in KEY_AMENITIES}
    s_lower = s.lower()
    count = len(re.findall(r'"[^"]+"', s))
    flags = {k: int(bool(re.search(pat, s_lower))) for k, pat in KEY_AMENITIES.items()}
    return count, flags

parsed = df['amenities'].apply(parse_amenities)
df['amenity_count'] = parsed.apply(lambda x: x[0])
for key in KEY_AMENITIES:
    df[key] = parsed.apply(lambda x: x[1][key])

print('Amenity count stats:')
print(df['amenity_count'].describe().round(1))
print('\nAmenity prevalence:')
print(df[list(KEY_AMENITIES)].mean().sort_values(ascending=False).round(3))

In [ ]:
# --- Host features -----------------------------------------------------------
def pct_to_float(s):
    if pd.isna(s): return np.nan
    return float(str(s).replace('%','').strip()) / 100.0

RESPONSE_TIME_MAP = {
    'within an hour': 1, 'within a few hours': 2,
    'within a day': 3, 'a few days or more': 4
}

df['host_response_rate_f']    = df['host_response_rate'].apply(pct_to_float)
df['host_acceptance_rate_f']  = df['host_acceptance_rate'].apply(pct_to_float)
df['host_response_time_ord']  = df['host_response_time'].str.lower().map(RESPONSE_TIME_MAP)
df['is_superhost']            = (df['host_is_superhost'] == 't').astype(int)
df['host_identity_verified_f']= (df['host_identity_verified'] == 't').astype(int)
df['is_instant_bookable']     = (df['instant_bookable'] == 't').astype(int)
df['has_license']             = df['license'].notna().astype(int)

# Fill medians for host numeric cols
for col in ['host_response_rate_f', 'host_acceptance_rate_f', 'host_response_time_ord']:
    df[col] = df[col].fillna(df[col].median())

print('Host features summary:')
print(df[['is_superhost','host_identity_verified_f','is_instant_bookable','has_license']].mean().round(3))

In [ ]:
# --- Review scores -----------------------------------------------------------
REVIEW_COLS = [
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value'
]
for col in REVIEW_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

print('Review scores (median):')
print(df[REVIEW_COLS].median().round(2))

In [ ]:
# --- Categoricals ------------------------------------------------------------
# Room type → ordinal (shared < private < entire)
ROOM_TYPE_MAP = {
    'Shared room': 0, 'Hotel room': 1,
    'Private room': 2, 'Entire home/apt': 3
}
df['room_type_ord'] = df['room_type'].map(ROOM_TYPE_MAP).fillna(1)

# Borough → OHE
borough_dummies = pd.get_dummies(df['neighbourhood_group_cleansed'], prefix='boro', drop_first=False)
df = pd.concat([df, borough_dummies], axis=1)

# Simplified property type (too many categories → group)
def simplify_property(s):
    if pd.isna(s): return 'Other'
    s = s.lower()
    if 'entire' in s: return 'Entire home'
    if 'private' in s: return 'Private room'
    if 'shared' in s: return 'Shared room'
    if 'hotel' in s: return 'Hotel room'
    return 'Other'

df['property_simplified'] = df['property_type'].apply(simplify_property)
prop_dummies = pd.get_dummies(df['property_simplified'], prefix='prop', drop_first=False)
df = pd.concat([df, prop_dummies], axis=1)

print('Room type distribution:')
print(df['room_type'].value_counts())

In [ ]:
# --- Fill remaining numeric NAs + log-price target --------------------------
for col in ['bedrooms', 'beds', 'reviews_per_month', 'estimated_occupancy_l365d',
            'estimated_revenue_l365d']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

df['log_price'] = np.log1p(df['price_numeric'])

print('Missing values after engineering:')
miss = df.isnull().sum()
print(miss[miss > 0])

---
## 4. EDA — Plotly Visualisations

In [ ]:
# Price distribution by borough
fig = px.box(
    df, x='neighbourhood_group_cleansed', y='price_numeric',
    color='neighbourhood_group_cleansed',
    title='Nightly Price Distribution by Borough (Nov 2025, STR listings)',
    labels={'neighbourhood_group_cleansed': 'Borough', 'price_numeric': 'Nightly price ($)'},
)
fig.update_yaxes(range=[0, df['price_numeric'].quantile(0.97)])
fig.update_layout(showlegend=False, height=450)
fig.write_html(str(REPORTS / 'price_by_borough.html'))
fig.show()

In [ ]:
# Interactive price heatmap
fig = px.density_mapbox(
    df,
    lat='latitude', lon='longitude',
    z='price_numeric',
    radius=14,
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=10,
    mapbox_style='open-street-map',
    title='NYC Airbnb Nightly Price Heatmap — Nov 2025',
    color_continuous_scale='Viridis',
    range_color=[0, df['price_numeric'].quantile(0.95)],
    labels={'price_numeric': 'Price ($)'},
)
fig.update_layout(height=650, margin={'r':0,'t':40,'l':0,'b':0})
fig.write_html(str(REPORTS / 'price_heatmap.html'))
print('Saved → reports/price_heatmap.html')
fig.show()

In [ ]:
# Listing scatter map coloured by price
fig = px.scatter_mapbox(
    df,
    lat='latitude', lon='longitude',
    color='price_numeric',
    size='accommodates',
    hover_name='neighbourhood_cleansed',
    hover_data={'price_numeric': True, 'room_type': True, 'accommodates': True,
                'dist_subway_km': ':.2f', 'crime_count_500m': True},
    color_continuous_scale='RdYlGn_r',
    range_color=[0, df['price_numeric'].quantile(0.95)],
    mapbox_style='open-street-map',
    title='NYC Airbnb Listings — Price & Size',
    zoom=10,
    center={'lat': 40.7128, 'lon': -74.0060},
)
fig.update_layout(height=650, margin={'r':0,'t':40,'l':0,'b':0})
fig.write_html(str(REPORTS / 'listings_map.html'))
fig.show()

In [ ]:
# Median price by neighbourhood (top 20)
nb_price = (df.groupby('neighbourhood_cleansed')['price_numeric']
              .median().sort_values(ascending=False).head(25).reset_index())
fig = px.bar(
    nb_price, x='price_numeric', y='neighbourhood_cleansed',
    orientation='h',
    title='Top 25 Neighbourhoods by Median Nightly Price',
    labels={'price_numeric': 'Median price ($)', 'neighbourhood_cleansed': ''},
    color='price_numeric', color_continuous_scale='Blues',
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.write_html(str(REPORTS / 'price_by_neighbourhood.html'))
fig.show()

In [ ]:
# Correlation heatmap of numeric features
CORR_COLS = [
    'price_numeric', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'amenity_count', 'availability_365', 'number_of_reviews',
    'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location',
    'dist_subway_km', 'dist_lirr_km', 'poi_count_500m', 'tourist_poi_500m',
    'crime_count_500m', 'is_superhost', 'is_instant_bookable',
    'host_acceptance_rate_f', 'host_response_rate_f',
]
CORR_COLS = [c for c in CORR_COLS if c in df.columns]
corr = df[CORR_COLS].corr(method='spearman').round(2)

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Spearman Correlation Matrix',
    text_auto=True,
)
fig.update_layout(height=700)
fig.write_html(str(REPORTS / 'correlation_matrix.html'))
fig.show()

In [ ]:
# Price vs key spatial features
fig = px.scatter(
    df, x='dist_subway_km', y='price_numeric',
    color='neighbourhood_group_cleansed',
    opacity=0.5, trendline='lowess',
    title='Price vs. Distance to Nearest Subway Station',
    labels={'dist_subway_km': 'Distance to subway (km)', 'price_numeric': 'Price ($)',
            'neighbourhood_group_cleansed': 'Borough'},
)
fig.update_yaxes(range=[0, df['price_numeric'].quantile(0.97)])
fig.write_html(str(REPORTS / 'price_vs_subway.html'))
fig.show()

In [ ]:
# Crime density vs price
fig = px.scatter(
    df, x='crime_count_500m', y='price_numeric',
    color='neighbourhood_group_cleansed',
    opacity=0.5, trendline='lowess',
    title='Price vs. Crime Density (arrests within 500m)',
    labels={'crime_count_500m': 'Arrests within 500m', 'price_numeric': 'Price ($)',
            'neighbourhood_group_cleansed': 'Borough'},
)
fig.update_yaxes(range=[0, df['price_numeric'].quantile(0.97)])
fig.write_html(str(REPORTS / 'price_vs_crime.html'))
fig.show()

---
## 5. Build & Save XGBoost Feature Matrix

In [ ]:
# All numeric features for XGBoost
FEATURE_COLS = [
    # Property
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'room_type_ord', 'amenity_count', 'minimum_nights',
    # Availability
    'availability_365',
    # Reviews
    'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    # Host
    'is_superhost', 'host_identity_verified_f', 'is_instant_bookable',
    'host_response_rate_f', 'host_acceptance_rate_f', 'host_response_time_ord',
    'has_license',
    # Amenity flags
    'has_wifi', 'has_kitchen', 'has_ac', 'has_gym', 'has_parking', 'has_pool',
    'has_washer', 'has_dryer', 'has_elevator', 'has_doorman', 'has_pets_allowed',
    # Spatial
    'dist_subway_km', 'near_subway', 'dist_lirr_km', 'lirr_count_1mi',
    'poi_count_500m', 'poi_count_1km', 'tourist_poi_500m', 'crime_count_500m',
    # Estimated occupancy/revenue (from InsideAirbnb)
    'estimated_occupancy_l365d', 'estimated_revenue_l365d',
]

# Borough OHE
boro_cols = [c for c in df.columns if c.startswith('boro_')]
prop_cols = [c for c in df.columns if c.startswith('prop_')]

ALL_FEATURES = [c for c in FEATURE_COLS + boro_cols + prop_cols if c in df.columns]
TARGET = 'price_numeric'

Xdf = df[['id'] + ALL_FEATURES + [TARGET, 'log_price',
          'latitude', 'longitude',
          'neighbourhood_cleansed', 'neighbourhood_group_cleansed']].copy()

# Final check: no NAs in feature columns
for col in ALL_FEATURES:
    if Xdf[col].isna().any():
        Xdf[col] = Xdf[col].fillna(Xdf[col].median())

print(f'XGBoost feature matrix: {Xdf.shape}')
print(f'Features: {len(ALL_FEATURES)}')
print(f'Target range: ${Xdf[TARGET].min():.0f} – ${Xdf[TARGET].max():.0f}  (median ${Xdf[TARGET].median():.0f})')

miss = Xdf[ALL_FEATURES].isnull().sum().sum()
print(f'Missing values in features: {miss}')

In [ ]:
# Feature importance preview via Spearman correlations with log_price
corrs = df[ALL_FEATURES].corrwith(df['log_price'], method='spearman').abs()
top_feats = corrs.sort_values(ascending=False).head(20)

fig = px.bar(
    x=top_feats.values, y=top_feats.index,
    orientation='h',
    title='Top 20 Features by |Spearman ρ| with log(price)',
    labels={'x': '|Spearman ρ|', 'y': 'Feature'},
    color=top_feats.values, color_continuous_scale='Blues',
)
fig.update_layout(height=550, yaxis={'categoryorder': 'total ascending'})
fig.write_html(str(REPORTS / 'feature_correlations.html'))
fig.show()

In [ ]:
# Save
out_path = PROCESSED / 'xgboost_features.csv'
Xdf.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Size: {out_path.stat().st_size / 1e6:.1f} MB')
print(f'\nFeature list ({len(ALL_FEATURES)} total):')
for i, f in enumerate(ALL_FEATURES, 1):
    print(f'  {i:2d}. {f}')